### Data Extraction from PDF

##### Installing the required libraries:

In [1]:
%pip install pymupdf4llm docling opendataloader_pdf marker-pdf unstructured pymupdf4llm pandas matplotlib seaborn scikit-learn pytesseract pillow

Note: you may need to restart the kernel to use updated packages.


In [2]:
# Standard imports used throughout the project
import os, sys, csv, io
import time, re
import json
import warnings
import pandas as pd
import seaborn as sns
from datetime import datetime
from pathlib import Path
from dotenv import load_dotenv
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

warnings.filterwarnings('ignore')

In [3]:
# Load API keys from .env
load_dotenv()  # reads .env in the project root

# Project root (works whether you open the notebook from the repo root or not)
ROOT = Path().resolve()
if not (ROOT / 'adapters').exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

CORPUS_DIR      = ROOT / 'corpus'
GROUND_TRUTH_DIR = ROOT / 'ground_truth'
OUTPUT_DIR      = ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)

print(f'Root: {ROOT}')
print(f'PDFs found: {list(CORPUS_DIR.glob("*.pdf"))}')

Root: /workspaces/PDF_Data_Extraction
PDFs found: [PosixPath('/workspaces/PDF_Data_Extraction/corpus/2_perfect-pitch-deck.pdf'), PosixPath('/workspaces/PDF_Data_Extraction/corpus/5_2025-integrated-annual-report.pdf'), PosixPath('/workspaces/PDF_Data_Extraction/corpus/3_NVCA-Model-Voting-Agreement.pdf'), PosixPath('/workspaces/PDF_Data_Extraction/corpus/4_SA-war-scanned.pdf'), PosixPath('/workspaces/PDF_Data_Extraction/corpus/1_Rapport-financier-semestriel-AFD-2025.pdf')]


In [4]:
# Corpus overview: page count, file size, rough text vs image ratio
import fitz  # PyMuPDF

pdf_files = sorted(CORPUS_DIR.glob('*.pdf'))

if not pdf_files:
    print('No PDFs found in corpus/. Add your 5 PDFs first.')
else:
    print(f'{'PDF':<35} {'Pages':>6} {'Size MB':>8} {'Text chars':>12} {'Images':>8} {'Type guess':>12}')
    print('-' * 85)
    
    for pdf_path in pdf_files:
        doc = fitz.open(str(pdf_path))
        n_pages = doc.page_count
        size_mb = pdf_path.stat().st_size / 1e6
        total_chars, total_images = 0, 0
        for page in doc:
            total_chars  += len(page.get_text())
            total_images += len(page.get_images())
        avg_chars = total_chars / n_pages if n_pages else 0
        
        if avg_chars < 1000 and total_images > 5:
            pdf_type = 'Image-heavy'
        elif avg_chars > 5000:
            pdf_type = 'Text-heavy'
        else:
            pdf_type = 'Mixed'
        print(f'{pdf_path.name:<35} {n_pages:>6} {size_mb:>8.1f} {total_chars:>12,} {total_images:>8} {pdf_type:>12}')
        doc.close()

PDF                                  Pages  Size MB   Text chars   Images   Type guess
-------------------------------------------------------------------------------------
1_Rapport-financier-semestriel-AFD-2025.pdf     53      0.7      150,986        2        Mixed
2_perfect-pitch-deck.pdf                16      3.8        1,000       15  Image-heavy
3_NVCA-Model-Voting-Agreement.pdf       23      0.3       83,336        0        Mixed
4_SA-war-scanned.pdf                    15      0.6       18,285       30        Mixed
5_2025-integrated-annual-report.pdf     38     12.7      162,006     1179        Mixed


##### Ground Truth:

In [40]:
# Generate a starting-point ground truth for one page 
GT_PDF  = pdf_files[5]   
GT_PAGE = 11            

doc  = fitz.open(str(GT_PDF))
page = doc[GT_PAGE - 1]

# extract text blocks from the page
blocks = page.get_text('blocks')

# Sort by vertical position then horizontal 
blocks_sorted = sorted(blocks, key=lambda b: (round(b[1]/20)*20, b[0]))

text_parts = []
for block in blocks_sorted:
    text = block[4].strip()

    if text:
        text_parts.append(text)

draft_text    = '\n\n'.join(b[4].strip() for b in blocks_sorted if b[4].strip())

doc.close()

In [41]:
# Print it for review
print(f'=== DRAFT ground truth: {GT_PDF.name}, page {GT_PAGE} ===')
print(draft_text[:5000])  # first 5000 chars
if len(draft_text) > 5000:
    print(f'... [{len(draft_text) - 5000} more chars — open the file to see all]')

# Auto-save (then manually edit the file) 
gt_dir = GROUND_TRUTH_DIR / GT_PDF.stem
gt_dir.mkdir(parents=True, exist_ok=True)
save_path = gt_dir / f'page_{GT_PAGE}.md'
save_path.write_text(draft_text, encoding='utf-8')
print(f'\nSaved to {save_path}')

=== DRAFT ground truth: 6_2025-integrated-annual-report.pdf, page 11 ===
INDIVIDUAL SHAREHOLDERS

THE SCC: A SHAREHOLDERS’ COMMUNICATION 
COMMITTEE TO TALK WITH THE GROUP

THEIR VOICES, OUR COMMITMENTS

First set up in 1987, the Shareholders’ Communication 
Committee (SCC) aims to establish regular exchanges 
between Air Liquide shareholders, the Chairman of 
the Board of Directors and Air Liquide Shareholders 
Relations team. Appointed for three-year terms, 
the twelve members meet three times a year.

ur individual shareholders are essential partners in our growth. Ambitious industrial 
projects, environmental impact, intergenerational transmission: their voices 
reǗect their expectations, their convictions, and their attachment to Air Liquide. 
The privileged relationship that unites us fuels our ambition to keep building the 
Group's future together.
O

Their remit is to:

• Convey the communication concerns and needs of 
individual shareholders;
• Make proposals to develop commu

In [11]:
# Extracting text from table
GT_PDF     = pdf_files[0]
TABLE_PAGE = 15

# Open PDF
with fitz.open(str(GT_PDF)) as doc:

    # PyMuPDF uses 0-based indexing
    page = doc[TABLE_PAGE - 1]

    # Print all text blocks with coordinates
    print(f'Text blocks on page {TABLE_PAGE}:\n')

    blocks = sorted(
        page.get_text("blocks"),
        key=lambda b: (b[1], b[0])   # sort top-to-bottom, left-to-right
    )

    for block in blocks:
        # Unpack block tuple
        x0, y0, x1, y1, text, block_no, block_type = block
        text = text.strip()

        if text:
            print(f'y={y0:6.0f}  x={x0:6.0f}  "{text[:80]}"')


# Manually reconstructed table from the PDF
correct_table = [
    # Header
    [
        'In thousands of euros',
        'Notes',
        '30 Jun 25',
        '31 Dec 24',
        'Change'
    ],

    # Rows
    ['Cash, due from central banks',
     ' ',
     '1,529,779',
     '863,504',
     '666,274'],

    ['Financial assets at fair value through profit or loss',
     '1',
     '3,868,930',
     '4,739,783',
     '(870,853)'],

    ['Hedging derivatives',
     '2',
     '2,586,028',
     '3,341,422',
     '(755,394)'],

    ['Financial assets at fair value through other comprehensive income',
     '3',
     '3,434,606',
     '2,273,869',
     '1,160,737'],

    ['Debt securities at amortised cost',
     '5',
     '4,031,971',
     '3,148,432',
     '883,539'],

    ['Financial assets at amortised cost',
     ' ',
     '53,009,695',
     '53,772,227',
     '(762,531)'],

    ['Loans and receivables due from credit institutions and equivalent at amortised cost',
     '5',
     '12,637,006',
     '13,303,340',
     '(666,334)'],

    ['On-demand',
     ' ',
     '932,054',
     '1,213,880',
     '(281,826)'],

    ['At maturity',
     ' ',
     '11,704,952',
     '12,089,460',
     '(384,508)'],

    ['Loans and receivables due from customers at amortised cost',
     '5',
     '40,372,689',
     '40,468,886',
     '(96,198)'],

    ['Other loans to customers',
     ' ',
     '40,372,689',
     '40,468,886',
     '(96,198)'],

    ['Of which calibration of the reserve account',
     ' ',
     '(1,047,780)',
     '(930,187)',
     '(117,592)'],

    ['Revaluation differences on interest rate-hedged portfolio',
     ' ',
     '14,045',
     '45,209',
     '(31,164)'],

    ['Current tax assets',
     ' ',
     '8,398',
     '5,966',
     '2,432'],

    ['Deferred tax assets',
     ' ',
     '27,419',
     '27,513',
     '(93)'],

    ['Accruals and other miscellaneous assets',
     '7',
     '3,466,979',
     '2,907,962',
     '559,017'],

    ['Accruals',
     ' ',
     '96,148',
     '53,516',
     '42,632'],

    ['Other assets',
     ' ',
     '3,370,831',
     '2,854,445',
     '516,386'],

    ['Equity stakes in companies accounted for by the equity method',
     '20',
     '158,014',
     '160,320',
     '(2,305)'],

    ['Fixed assets property, plant and equipment',
     '8',
     '902,952',
     '858,161',
     '44,791'],

    ['Intangible assets',
     '8',
     '190,124',
     '182,597',
     '7,527'],

    ['TOTAL ASSETS',
     ' ',
     '73,228,942',
     '72,326,964',
     '901,978']
]

correct_table1 = [

    # Header
    [
        'In thousands of euros',
        'Notes',
        '30 Jun 25',
        '31 Dec 24',
        'Change'
    ],

    # Rows
    ['Financial liabilities at fair value through profit or loss',
     '1',
     '95,270',
     '481,623',
     '(386,353)'],

    ['Hedging derivatives',
     '2',
     '4,159,400',
     '3,662,740',
     '496,660'],

    ['Financial liabilities at amortised cost',
     ' ',
     '54,396,791',
     '53,477,032',
     '919,759'],

    ['Debt securities in issue at amortised cost',
     '9',
     '54,362,989',
     '53,465,351',
     '897,639'],

    ['Interbank market securities',
     ' ',
     '2,012,968',
     '809,211',
     '1,203,757'],

    ['Bonds',
     ' ',
     '52,350,022',
     '52,656,140',
     '(306,118)'],

    ['Debts to credit institutions and equivalent at amortised cost',
     '9',
     '32,210',
     '9,556',
     '22,654'],

    ['On-demand',
     ' ',
     '21,648',
     '9,016',
     '12,633'],

    ['At maturity',
     ' ',
     '10,561',
     '540',
     '10,021'],

    ['Debts to customers at amortised cost',
     '9',
     '1,592',
     '2,125',
     '(533)'],

    ['Current tax liabilities',
     ' ',
     '7,470',
     '14,441',
     '(6,971)'],

    ['Deferred tax liabilities',
     ' ',
     '10,855',
     '13,872',
     '(3,017)'],

    ['Accruals and other miscellaneous liabilities',
     '7',
     '3,044,424',
     '3,330,294',
     '(285,870)'],

    ['Allocated public funds',
     ' ',
     '94,813',
     '87,110',
     '7,704'],

    ['Other liabilities',
     ' ',
     '2,949,610',
     '3,243,184',
     '(293,574)'],

    ['Provisions',
     '10',
     '803,953',
     '882,354',
     '(78,401)'],

    ['Of which calibration of the reserve account',
     ' ',
     '333,431',
     '285,324',
     '48,107'],

    ['Subordinated debt',
     '11',
     '987,897',
     '842,617',
     '145,280'],

    ['TOTAL DEBTS',
     ' ',
     '63,506,058',
     '62,704,972',
     '801,085'],

    ['Equity Group share',
     '(Tab 1)',
     '9,531,780',
     '9,422,346',
     '109,433'],

    ['Provisions and related retained earnings',
     ' ',
     '5,177,999',
     '5,177,999',
     '-'],

    ['Consolidated retained earnings and other',
     ' ',
     '4,068,114',
     '3,786,818',
     '281,296'],

    ['Gains and losses recognised in other comprehensive income',
     ' ',
     '131,493',
     '113,918',
     '17,575'],

    ['Earnings for the period',
     ' ',
     '154,174',
     '343,612',
     '(189,438)'],

    ['Non-controlling interests',
     '(Tab 1)',
     '191,105',
     '199,646',
     '(8,541)'],

    ['Total equity',
     ' ',
     '9,722,885',
     '9,621,992',
     '100,893'],

    ['TOTAL LIABILITIES',
     ' ',
     '73,228,942',
     '72,326,964',
     '901,978']
]

# Create output directory
gt_dir = GROUND_TRUTH_DIR / GT_PDF.stem
gt_dir.mkdir(parents=True, exist_ok=True)

# Save CSV
csv_path = gt_dir / 'page_15.csv'

with open(csv_path, 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerows(correct_table)
    writer.writerows(correct_table1)

print(f'\nSaved ground truth table to:\n{csv_path}')

Text blocks on page 15:

y=    23  x=    57  "AFD – 2025 half-year financial report 
15"
y=    61  x=    71  "Balance sheet at 30 June 2025"
y=    96  x=    73  "Assets"
y=   113  x=    73  "In thousands of euros
Notes
30 Jun 25
31 Dec 24
Change"
y=   127  x=    73  "Cash, due from central banks
1,529,779
863,504
666,274"
y=   139  x=    73  "Financial assets at fair value through profit or loss
1
3,868,930
4,739,783
(870"
y=   151  x=    73  "Hedging derivatives
2
2,586,028
3,341,422
(755,394)"
y=   163  x=    73  "Financial assets at fair value through other comprehensive income
3
3,434,606
2,"
y=   175  x=    73  "Debt securities at amortised cost
5
4,031,971
3,148,432
883,539"
y=   187  x=    73  "Financial assets at amortised cost
53,009,695
53,772,227
(762,531)"
y=   199  x=    73  "Loans and receivables due from credit institutions and equivalent at 
amortised "
y=   222  x=    80  "On-demand
932,054
1,213,880
(281,826)"
y=   234  x=    80  "At maturity
11,704,952
12,089,460
(38

##### Ground Truth for presentations & images

In [35]:
import pytesseract
from pytesseract import Output
from PIL import Image

GT_PDF  = pdf_files[5]
GT_PAGE = 11

doc  = fitz.open(str(GT_PDF))
page = doc[GT_PAGE - 1]

# Convert a page to image
pix = page.get_pixmap(dpi=300)
img = Image.open(io.BytesIO(pix.tobytes("png")))

In [36]:
# OCR extraction
ocr_data = pytesseract.image_to_data( img, output_type=pytesseract.Output.DICT)
doc.close()

In [37]:
lines = {}

for i in range(len(ocr_data["text"])):
    text = ocr_data["text"][i].strip()

    if not text:
        continue

    x = ocr_data["left"][i]
    y = ocr_data["top"][i]

    # group words into same line
    y_key = round(y / 15) * 15

    lines.setdefault(y_key, []).append((x, text))

# build pseudo-blocks
blocks = []

for y_key in sorted(lines.keys()):
    words = sorted(lines[y_key], key=lambda w: w[0])  # left → right

    line_text = " ".join(w[1] for w in words)

    x0 = words[0][0]

    blocks.append((x0, y_key, x0, y_key, line_text))


blocks_sorted = sorted(
    blocks,
    key=lambda b: (round(b[1] / 20) * 20, b[0])
)

draft_text = "\n\n".join(
    b[4] for b in blocks_sorted if b[4].strip()
)

print(draft_text)

INDIVIDUAL SHAREHOLDERS

SHAREHOLDERS’

THE SCC: A COMMUNICATION

COMMITTEE TO TALK WITH THE GROUP

THEIR VOICES, OUR COMMITMENTS

ur individual shareholders are essential partners in our growth. Ambitious industrial

projects, environmental impact, intergenerational transmission: their voices

reflect their expectations, their convictions, and their attachment to Air Liquide.

The privileged relationship that unites us fuels our ambition to keep building the

Group's future together.

Air Liquide is positioned

on future-defining technologies.

I'm convinced

| became an Air Liquide shareholder thanks to my father. He'd

I'm investing ina to get me a was

been trying started for long time, but | a little

intimidated by the stock market. When daughter born,

my was

solid company.

he made it concrete: he of to buy

very gave me a sum money

Air Liquide shares for her. So my daughter became a shareholder

Air Liquide was my very first

at just six months old, and | became one as well 

In [38]:
# Auto-save (then manually edit the file) 
gt_dir = GROUND_TRUTH_DIR / GT_PDF.stem
gt_dir.mkdir(parents=True, exist_ok=True)
save_path = gt_dir / f'page_{GT_PAGE}.md'
save_path.write_text(draft_text, encoding='utf-8')
print(f'\nSaved to {save_path}')


Saved to /workspaces/PDF_Data_Extraction/ground_truth/6_2025-integrated-annual-report/page_11.md


Ground Table for charts:

In [29]:
from collections import defaultdict

GT_PDF  = pdf_files[5]
GT_PAGE = 10

doc  = fitz.open(str(GT_PDF))
page = doc[GT_PAGE - 1]

# Extract text blocks 
raw_blocks = page.get_text("blocks")   # (x0, y0, x1, y1, text, block_no, block_type)
page_width = page.rect.width

doc.close()

In [30]:
# Column-aware sorting helper

N_COLS = 2   # for two-column pages

def col_key(block, page_width, n_cols=N_COLS):
    col = int(block[0] / (page_width / n_cols))
    return (col, round(block[1] / 10) * 10, block[0])

# Deduplicate & sort blocks
seen   = set()
blocks = []

for block in sorted(raw_blocks, key=lambda b: col_key(b, page_width)):
    text = block[4].strip()
    norm = re.sub(r"\s+", " ", text).lower()
    if text and norm not in seen:
        seen.add(norm)
        blocks.append(block)


# Sort blocks (column → top → bottom)
blocks_sorted = sorted(
    blocks,
    key=lambda b: col_key(b, page_width)
)

# Build draft text
text_parts = []

for block in blocks_sorted:
    text = block[4].strip()
    if text:
        text_parts.append(text)

draft_text = "\n\n".join(text_parts)


In [31]:
# Print for review
print(f'=== DRAFT ground truth: {GT_PDF.name}, page {GT_PAGE} ===')
print(draft_text[:3000])

if len(draft_text) > 3000:
    print(f'... [{len(draft_text) - 3000} more chars — open file to see all]')

# Auto-save ground truth
gt_dir = GROUND_TRUTH_DIR / GT_PDF.stem
gt_dir.mkdir(parents=True, exist_ok=True)
save_path = gt_dir / f'page_{GT_PAGE}.md'
save_path.write_text(
    draft_text,
    encoding='utf-8'
)

print(f'\nSaved to:\n{save_path}')

=== DRAFT ground truth: 6_2025-integrated-annual-report.pdf, page 10 ===
OUR SHAREHOLDERS

PARTNERS IN PERFORMANCE

At Air Liquide, the structure of our shareholder 
base is a major asset. It is built on a unique 
complementarity between our institutional 
investors, who bring ǖnancial strength, international diversity 
and high standards in governance and sustainability, and our 
individual shareholders, the long-standing pillars of our 
shareholder stability.

Together, they form a distinctive foundation built on strong values of 
transparency and dialogue, which are essential to understanding our 
performance and strategy. This relationship of mutual trust, built over time, 
facilitates investment decisions and drives the major strategic choices that 
support our growth, such as the recent acquisition of DIG Airgas in South Korea. 
It also enables us to create value in a consistent, responsible and sustainable 
way for all.

Aude Rodriguez,

VP HEAD OF INVESTOR RELATIONS AT AIR LIQU

## PDF Extraction

**Goal:** Extract text, tables, images, and metadata from PDF files.

1. PyMuPDF4llm

In [5]:
import pymupdf4llm, fitz, re

TEST_PDF = pdf_files[0]  # start with the financial statement

t0 = time.perf_counter()
md_text = pymupdf4llm.to_markdown(str(TEST_PDF))
elapsed = time.perf_counter() - t0

doc     = fitz.open(str(TEST_PDF))
n_pages = doc.page_count
doc.close()

print(f'Extracted {len(md_text):,} chars from {n_pages} pages in {elapsed:.1f}s')
print(f'Pages/sec: {n_pages/elapsed:.1f}')
print()
print('=== First 1500 characters of output ===')
print(md_text[:1500])

2026-05-28 07:28:42.964324769 [W:onnxruntime:Default, device_discovery.cc:133 GetPciBusId] Skipping pci_bus_id for PCI path at "/sys/devices/LNXSYSTM:00/LNXSYBUS:00/PNP0A03:00/device:07/VMBUS:01/5620e0c7-8062-4dce-aeb7-520c7ef76171" because filename "5620e0c7-8062-4dce-aeb7-520c7ef76171" did not match expected pattern of [0-9a-f]+:[0-9a-f]+:[0-9a-f]+[.][0-9a-f]+


=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=0/1.
OCR on page.number=25/26.

Extracted 155,525 chars from 53 pages in 26.3s
Pages/sec: 2.0

=== First 1500 characters of output ===
## **Half-year financial report** 

**30 June 2025** 

AFD – 2025 half-year financial report 

2 

**Contents** A. Management report ............................................................................................................................ 4 1. AFD Group activities ....................................................................................................................... 4 2. Recent changes and outlook .......................................................................................................... 6 2.1. Crises in several countries ................................................................................................... 6 2.2. Refinancing and liquidity ..................................................................

In [6]:
# ── Heading extraction from the Markdown output ──────────────────────────────
headings = []
for line in md_text.splitlines():
    m = re.match(r'^(#{1,6})\s+(.*)', line)
    if m:
        headings.append((len(m.group(1)), m.group(2).strip()))

print(f'Found {len(headings)} headings:')
for level, title in headings[:20]:
    print(f'  {"  " * (level-1)}H{level}: {title}')

Found 168 headings:
    H2: **Half-year financial report**
    H2: **A. Management report**
    H2: **1. AFD Group activities**
    H2: **Approvals**
    H2: **Activities in foreign countries**
    H2: On its own behalf
    H2: On behalf of third parties
    H2: **Activities in the French Overseas Departments and Collectivities**
    H2: On its own behalf
    H2: Loans guaranteed by the French State
    H2: **Proparco’s activity**
    H2: **Disbursements**
    H2: **Activities in foreign countries**
    H2: On its own behalf
    H2: On behalf of third parties
    H2: **Activities in the French Overseas Departments and Collectivities**
    H2: On its own behalf
    H2: Loans guaranteed by the French State
    H2: **Proparco’s activity**
    H2: **2. Recent changes and outlook**


In [7]:
# ── Table extraction from Markdown pipe tables ───────────────────────────────
def parse_md_tables(md):
    tables, current = [], []
    for line in md.splitlines():
        s = line.strip()
        if s.startswith('|') and s.endswith('|'):
            cells = [c.strip() for c in s.strip('|').split('|')]
            if all(re.match(r'^[-:]+$', c) for c in cells if c):
                continue  # separator row
            current.append(cells)
        else:
            if current: tables.append(current); current = []
    if current: tables.append(current)
    return tables

tables = parse_md_tables(md_text)
print(f'Found {len(tables)} tables')
for i, t in enumerate(tables[:3]):
    print(f'\nTable {i+1} ({len(t)} rows x {len(t[0]) if t else 0} cols):')
    for row in t[:5]:
        print('  | ' + ' | '.join(str(c)[:20] for c in row))


Found 55 tables

Table 1 (6 rows x 4 cols):
  | **Maturity** | **Currency** | **Nominal in currenc | **EUR equivalent**
  | 20/01/2035 | EUR | 2,000,000,000 | 2,000,000,000
  | 03/04/2040 | EUR | 1,000,000,000 | 1,000,000,000
  | 30/09/2030 | EUR | 1,500,000,000 | 1,500,000,000
  | 16/01/2030 | USD | 1,000,000,000 | 970,873,786

Table 2 (6 rows x 4 cols):
  | **Maturity** | **Currency** | **Nominal in currenc | **EUR equivalent**
  | 25/05/2036 | EUR | 100,000,000 | 100,000,000
  | 02/03/2037 | EUR | 150,000,000 | 150,000,000
  | 16/01/2030 | USD | 100,000,000 | 92,412,901
  | 16/01/2030 | USD | 100,000,000 | 91,911,765

Table 3 (4 rows x 4 cols):
  | **Maturity** | **Currency** | **Nominal in currenc | **EUR equivalent**
  | 27/02/2040 | AUD | 40,000,000 | 24,345,709
  | 15/07/2026 | TRY | 1,500,000,000 | 40,760,870
  | 06/03/2028 | TRY | 1,500,000,000 | 39,215,686


In [8]:
# ── Save output for the scoring step ─────────────────────────────────────────
out_dir = OUTPUT_DIR / 'pymupdf4llm' / TEST_PDF.stem
out_dir.mkdir(parents=True, exist_ok=True)
(out_dir / 'text.md').write_text(md_text, encoding='utf-8')

tables_dir = out_dir / 'tables'
tables_dir.mkdir(exist_ok=True)
import csv as _csv
for i, t in enumerate(tables):
    with open(tables_dir / f'table_{i+1:02d}.csv', 'w', newline='', encoding='utf-8') as f:
        _csv.writer(f).writerows(t)

print(f'Saved to {out_dir}')

Saved to /workspaces/PDF_Data_Extraction/outputs/pymupdf4llm/1_Rapport-financier-semestriel-AFD-2025
